# 02 - Merge, Filter and Build Analysis Dataset
Project: MetS Risk Score - Multimodel prediction in young adults
Input: 15 raw NHANES P_ CSVs from data/raw
Output: Single analysis ready dataset

In [1]:
# Cell 2: Mount Drive and Define Paths
from google.colab import drive
drive.mount('/content/drive')

import os

Drive_root = '/content/drive/MyDrive/mets-risk-score/'
data_raw = os.path.join(Drive_root, 'data/raw/')
data_proc = os.path.join(Drive_root, 'data/processed/')
fig_dir = os.path.join(Drive_root, 'outputs/figures/')

print("Drive mounted sucessfully")
print(f"Raw Data: {data_raw}")
print(f"Processed: {data_proc}")

Mounted at /content/drive
Drive mounted sucessfully
Raw Data: /content/drive/MyDrive/mets-risk-score/data/raw/
Processed: /content/drive/MyDrive/mets-risk-score/data/processed/


In [2]:
# Cell 3: Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)


In [3]:
# Cell 4: Load all 15 tables from drive

table_names = [
    'DEMO',    # Demographics
    'BIOPRO',  # Secondary biochemistry
    'CBC',     # Complete blood count
    'DR1TOT',  # Dietary recall day 1
    'DR2TOT',  # Dietary recall day 2
    'PAQ',     # Physical activity
    'SLQ',     # Sleep
    'DPQ',     # Mental health (PHQ-9)
    'SMQ',     # Smoking
    'ALQ',     # Alcohol
    'BMX',     # Body measures
    'TRIGLY',  # Triglycerides (target component)
    'HDL',     # HDL cholesterol (target component)
    'GLU',     # Fasting glucose (target component)
    'BPX',     # Blood pressure (target component)
]

# For clean and aligned columns, looks like a proper table
print(f"{'Table':<10} {'Rows':>8} {'Columns':>10}")

tables = {}

for name in table_names:
  path = os.path.join(data_raw, f"{name}.csv")

  if not os.path.exists(path):
    print(f"{name:<10} file not found at {path}")
    continue

  df = pd.read_csv(path, low_memory= False)

  # Ensure SEQN is an integer for clean merging
  df['SEQN'] = df['SEQN'].astype(int)

  tables[name] = df
  print(f"{name:<10} {len(df):>8,} {len(df.columns):>10}")

print(f"\n {len(tables)} tables loaded into memory")

Table          Rows    Columns
DEMO         15,560         29
BIOPRO       10,409         41
CBC          13,772         22
DR1TOT       14,300        168
DR2TOT       14,300         85
PAQ           9,693         17
SLQ          10,195         11
DPQ           8,965         11
SMQ          11,137         16
ALQ           8,965         10
BMX          14,300         22
TRIGLY        5,090         10
HDL          12,198          3
GLU           5,090          4
BPX          11,656         12

 15 tables loaded into memory


In [4]:
# Cell 5: Merge all tables on SEQN

# Start with demographics as the base
merged = tables['DEMO'].copy()
print(f"Base  (DEMO): {len(merged):>6,} participats")

# left join each additional table
for name in table_names:
  if name == 'DEMO':
    continue

  df = tables[name]

  # Identify columns that would duplicate (other than SEQN)
  # Add suffix to avoid column name collisions across tables
  before_cols = len(merged.columns)
  merged = merged.merge(df, on = 'SEQN', how = 'left', suffixes = ('', f'_{name}'))
  added = len(merged.columns) - before_cols

  print(f"   + {name:<10}     : {len(merged):>6,} rows | +{added} columns")

Base  (DEMO): 15,560 participats
   + BIOPRO         : 15,560 rows | +40 columns
   + CBC            : 15,560 rows | +21 columns
   + DR1TOT         : 15,560 rows | +167 columns
   + DR2TOT         : 15,560 rows | +84 columns
   + PAQ            : 15,560 rows | +16 columns
   + SLQ            : 15,560 rows | +10 columns
   + DPQ            : 15,560 rows | +10 columns
   + SMQ            : 15,560 rows | +15 columns
   + ALQ            : 15,560 rows | +9 columns
   + BMX            : 15,560 rows | +21 columns
   + TRIGLY         : 15,560 rows | +9 columns
   + HDL            : 15,560 rows | +2 columns
   + GLU            : 15,560 rows | +3 columns
   + BPX            : 15,560 rows | +11 columns


In [5]:
# Cell 6: Filter to Young Adults Age 18-45
# RIDAGEYR = age in years at screening (From DEMO table)

before = len(merged)
age_filtered = merged[(merged['RIDAGEYR'] >=18) & (merged['RIDAGEYR'] <=45)].copy()
after = len(age_filtered)
removed = before - after

print(f"   Before filter : {before:>6,} participants")
print(f"   After filter  : {after:>6,} participants")
print(f"   Removed       : {removed:>6,} (outside 18–45 age range)")

   Before filter : 15,560 participants
   After filter  :  4,142 participants
   Removed       : 11,418 (outside 18–45 age range)


In [6]:
# Cell 7: Apply Exclusion Criteria

# We exclude participants who would make the analysis unreliable or biased.
# Each exclusion is documented with a reason — this becomes your
# "Study Population" paragraph in the methods section.
#
# RIDSTATR = interview/exam status
#   1 = interviewed only (no physical exam — missing lab values)
#   2 = interviewed AND examined (complete data) ← keep these only
#
# RIAGENDR = gender (1=Male, 2=Female)
# RIDEXPRG = pregnancy status (1=pregnant) ← exclude pregnant participants
#            because pregnancy significantly alters metabolic markers

cohort = age_filtered.copy()
exclusion_log =[]

def exclude(df, condition, reason):
  """Apply Exclusion, log how many were removed, reture filterd df"""
  before = len(df)
  df_filtered = df[~condition].copy()
  after = len(df_filtered)
  removed = before - after
  exclusion_log.append({
      'Reason': reason,
      'Removed': removed,
      'Remaining': after
  })
  return df_filtered

# Exclusion 1: Exam-only participants (no lab data)
cohort = exclude(
    cohort,
    cohort['RIDSTATR'] != 2,
    'No physical examination(interview only)'
)

# Exclusion 2: Pregnenet Participants
# RIDEXPRG = 1 means confirmed pregnent at exam
cohort = exclude(
    cohort,
    cohort['RIDEXPRG'] == 1,
    'Pregnent at the time of examination'
)

# Print Exclusive table
print(f"{'Step':<5} {'Reason':<45} {'Removed':>8} {'Remaining':>10}")
print("─" * 72)

start = len(age_filtered)
print(f"{'0':<5} {'Age-filtered cohort (18-45)':<45} {'—':>8} {start:>10,}")

for i, row in enumerate(exclusion_log, 1):
    print(f"{i:<5} {row['Reason']:<45} {row['Removed']:>8,} {row['Remaining']:>10,}")

print("─" * 72)
print(f"\n Final cohort after exclusions: {len(cohort):,} participants")


Step  Reason                                         Removed  Remaining
────────────────────────────────────────────────────────────────────────
0     Age-filtered cohort (18-45)                          —      4,142
1     No physical examination(interview only)            305      3,837
2     Pregnent at the time of examination                 87      3,750
────────────────────────────────────────────────────────────────────────

 Final cohort after exclusions: 3,750 participants


In [7]:
# Cell 8: Cohort Summary
# Quick demographic screen shot of who's in the final cohort

print(f"Total Participants: {len(cohort):,}")
print(f"Total Variables: {cohort.shape[1]:,}")

# Sex distribution
# RIAGENDR: 1 = Male, 2 = Female
sex_counts = cohort['RIAGENDR'].value_counts()
print(f"Male   : {sex_counts.get(1.0, 0):,} ({sex_counts.get(1.0, 0)/len(cohort)*100:.1f}%)")
print(f"Female : {sex_counts.get(2.0, 0):,} ({sex_counts.get(2.0, 0)/len(cohort)*100:.1f}%)")

# Race/Ethnicity
# RIDRETH3: 1=Mexican American, 2=Other Hispanic, 3=Non-Hispanic White,
#           4=Non-Hispanic Black, 6=Non-Hispanic Asian, 7=Other/Multi
race_map = {
    1.0: 'Mexican American',
    2.0: 'Other Hispanic',
    3.0: 'Non-Hispanic White',
    4.0: 'Non-Hispanic Black',
    6.0: 'Non-Hispanic Asian',
    7.0: 'Other/Multiracial'
}
print(f"\n   Race/Ethnicity:")
for code, label in race_map.items():
    n = (cohort['RIDRETH3'] == code).sum()
    pct = n / len(cohort) * 100
    print(f"   {label:<25} : {n:,} ({pct:.1f}%)")

# Age
print(f" Mean ± SD : {cohort['RIDAGEYR'].mean():.1f} ± {cohort['RIDAGEYR'].std():.1f}")
print(f" Median    : {cohort['RIDAGEYR'].median():.1f}")

# Missingness overview
total_cells = cohort.shape[0] * cohort.shape[1]
missing_cells = cohort.isnull().sum().sum()
print(f"\n  Overall missingness : {missing_cells/total_cells*100:.1f}% of all values")

Total Participants: 3,750
Total Variables: 447
Male   : 1,812 (48.3%)
Female : 1,938 (51.7%)

   Race/Ethnicity:
   Mexican American          : 551 (14.7%)
   Other Hispanic            : 404 (10.8%)
   Non-Hispanic White        : 1,119 (29.8%)
   Non-Hispanic Black        : 959 (25.6%)
   Non-Hispanic Asian        : 488 (13.0%)
   Other/Multiracial         : 229 (6.1%)
 Mean ± SD : 31.2 ± 8.4
 Median    : 31.0

  Overall missingness : 30.0% of all values


In [8]:
# Cell 9: Save final cohort to data/processed

save_path = os.path.join(data_proc, 'merged_cohort.csv')
cohort.to_csv(save_path, index = False)
size_mb = os.path.getsize(save_path) / (1024*1024)

print(f" Path: {save_path}")

 Path: /content/drive/MyDrive/mets-risk-score/data/processed/merged_cohort.csv


In [9]:
# Cell 10: Compute MSSS Target & Seperate Predictors

mass_cols = ['BPXOSY1', 'BPXOSY2', 'BPXODI1', 'BPXODI2',
             'BMXWAIST', 'LBXTR', 'LBDHDD', 'LBXGLU',
             'RIAGENDR', 'RIDRETH3']

missing_mass_cols = [c for c in mass_cols if c not in cohort.columns]
if missing_mass_cols:
  print(f"Missing columns: {missing_mass_cols}")
else:
  print(f"All required columns present")

# Average BP ratings (NHANES multiple mesurments)
cohort['SYSTOLIC_AVG']  = cohort[['BPXOSY1', 'BPXOSY2']].mean(axis=1)
cohort['DIASTOLIC_AVG'] = cohort[['BPXODI1', 'BPXODI2']].mean(axis=1)

# Identify rows with complete MSSS inputs

msss_input_cols = ['SYSTOLIC_AVG', 'DIASTOLIC_AVG', 'BMXWAIST', 'LBXTR', 'LBDHDD', 'LBXGLU']
has_complete_msss_input = cohort[msss_input_cols].notna().all(axis= 1)

print(f"   Participants with ALL 5 components present: {has_complete_msss_input.sum():,} / {len(cohort):,}")
print(f"   ({has_complete_msss_input.sum() / len(cohort) * 100:.1f}% of current cohort)")



All required columns present
   Participants with ALL 5 components present: 1,499 / 3,750
   (40.0% of current cohort)


In [10]:
# Cell 11a: Check 18-19 Age Group Size

age_18_19 = cohort[(cohort['RIDAGEYR'] >= 18) & (cohort['RIDAGEYR'] <= 19)]
age_20_45 = cohort[(cohort['RIDAGEYR'] >= 20) & (cohort['RIDAGEYR'] <= 45)]

print(f"   18-19 years (adolescent formula) : {len(age_18_19):,}")
print(f"   20-45 years (adult formula)      : {len(age_20_45):,}")

# Check how many 18-19 year-olds have complete MSSS inputs
msss_input_cols = ['SYSTOLIC_AVG', 'DIASTOLIC_AVG', 'BMXWAIST', 'LBXTR', 'LBDHDD', 'LBXGLU']
age_18_19_complete = age_18_19[msss_input_cols].notna().all(axis=1).sum()
age_20_45_complete = age_20_45[msss_input_cols].notna().all(axis=1).sum()

print(f"   18-19 group : {age_18_19_complete:,} of {len(age_18_19):,}")
print(f"   20-45 group : {age_20_45_complete:,} of {len(age_20_45):,}")
print(f"\n   18-19 with complete data as % of total 1,499: "
      f"{age_18_19_complete/1499*100:.1f}%")

   18-19 years (adolescent formula) : 421
   20-45 years (adult formula)      : 3,329
   18-19 group : 144 of 421
   20-45 group : 1,355 of 3,329

   18-19 with complete data as % of total 1,499: 9.6%


In [11]:
# Cell 11b: Download CDC BMI for Age LMS Reference Table

# Why we need it?
# The Gurka-DeBoer adolescent MSSS formula takes bmiZScore as input,
# not raw BMI. So for our 18-19 year-olds we must first convert their
# BMI into a z-score using these CDC reference values before plugging
# into the MSSS formula.

# THE FORMULA (confirmed from CDC and multiple clinical trial SAPs):
#   If L ≠ 0:  Z = [((BMI/M)^L) - 1] / (L * S)
#   If L = 0:  Z = ln(BMI/M) / S
#
# Source: https://www.cdc.gov/growthcharts/cdc-data-files.htm

import requests

LMS_URL = "https://www.cdc.gov/growthcharts/data/zscore/bmiagerev.csv"
LMS_PATH = os.path.join(data_raw, 'cdc_bmi_lms.csv')

if os.path.exists(LMS_PATH):
  print("CDC LMS table is in drive already")
else:
  print("Donwloading the CDC BMI for age LMS Table")
  response = requests.get(LMS_URL, timeout=30)
  if response.status_code == 200:
    with open(LMS_PATH, 'wb') as f:
      f.write(response.content)
    print("Saved")
  else:
    print(f"HTTP{response.status_code}")

# load & Inspect
lms = pd.read_csv(LMS_PATH)
print(f"Shape:{lms.shape}")
print(f"Columns:{lms.columns.tolist()}")
print(lms.head(3).to_string(index = False))


CDC LMS table is in drive already
Shape:(439, 15)
Columns:['Sex', 'Agemos', 'L', 'M', 'S', 'P3', 'P5', 'P10', 'P25', 'P50', 'P75', 'P85', 'P90', 'P95', 'P97']
Sex Agemos            L           M           S          P3          P5         P10         P25         P50         P75         P85         P90         P95         P97
  1     24  -2.01118107 16.57502768 0.080592465 14.52095333 14.73731947 15.09032827 15.74164233 16.57502768 17.55718781 18.16219473 18.60948128 19.33801062 19.85985812
  1   24.5 -1.982373595 16.54777487 0.080127429 14.50347667 14.71929257 15.07117474 15.71962876 16.54777487 17.52129279 18.11954923 18.56110634 19.27889813 19.79194014
  1   25.5 -1.924100169 16.49442763 0.079233994 14.46882381 14.68360841 15.03335725 15.67634464 16.49442763 17.45135039 18.03668013 18.46729593 19.16465965 19.66102345


In [12]:
# Cell 11c: compute BMI Z-score for Adolescents (Age 18-19)

# HOW THIS WORKS:
# 1. For each participant aged 18-19, we look up their L, M, S values
#    from the CDC table using their sex and age in months
# 2. We apply the LMS formula to convert raw BMI → z-score
# 3. This z-score then feeds into the Gurka-DeBoer adolescent MSSS formula

# NHANES variables we need:
#   BMXBMI   = raw BMI (kg/m²)
#   RIDAGEYR = age in years
#   RIDAGEMN = age in months (more precise — needed for LMS lookup)
#   RIAGENDR = sex (1=Male, 2=Female)

# CDC LMS table coding:
#   Sex: 1 = Male, 2 = Female (matches NHANES RIAGENDR directly)
#   Age: listed at half-month point (e.g. 216.5 = 216-216.99 months)

def compute_bmi_zscore(bmi, age_months, sex, lms_table):
  """
    Compute CDC BMI-for-age z-score using LMS method.

    Parameters:
    -----------
    bmi        : float - raw BMI in kg/m²
    age_months : float - age in months
    sex        : int   - 1=Male, 2=Female (NHANES coding)
    lms_table  : pd.DataFrame - CDC LMS reference table

    Returns:
    --------
    float - BMI z-score, or NaN if lookup fails
  """
  if pd.isna(bmi) or pd.isna(age_months):
    return np.nan

  # Round age to nearest half-month for CDC table lookup
  # CDC ages are at 0.5 month midpoints (e.g. 216.5, 217.5)

  age_rounded = round(age_months) + 0.5

  # Filter LMS table for matching sex and age
  row = lms_table[
      (lms_table['Sex'] == sex)&
      (lms_table['Agemos'] == age_rounded)
  ]

  if row.empty:
    return np.nan
  L = row['L'].values[0]
  M = row['M'].values[0]
  S = row['S'].values[0]

  # Apply LMS formula
  if L != 0:
      z = (((bmi / M) ** L) - 1) / (L * S)
  else:
      z = np.log(bmi / M) / S

  return z

# Apply to 18-19 years old only

print("Computing BMI Z-scores for 18-19 year-olds")

# Check required columns
required = ['BMXBMI', 'RIDAGEMN', 'RIAGENDR']
missing = [c for c in required if c not in cohort.columns]
if missing:
  print(f"Missing columns: {missing}")
else:
  print("All required cpolumns present")

# Initialize BMI Z-score column with NaN for everyone
cohort['BMI_ZSCORE'] = np.nan
# Identify 18-19 years old
mask_adolescent = (cohort['RIDAGEYR'] >= 18) & (cohort['RIDAGEYR'] <= 19)
# compute Z-score by row by row for adolescents
adolescent_idx = cohort[mask_adolescent].index

for idx in adolescent_idx:
  bmi = cohort.loc[idx, 'BMXBMI']
  age_months = cohort.loc[idx, 'RIDAGEMN']
  sex = cohort.loc[idx, 'RIAGENDR']
  cohort.loc[idx, 'BMI_ZSCORE'] = compute_bmi_zscore(
            bmi, age_months, int(sex), lms
        )

# Results
computed = cohort.loc[mask_adolescent, 'BMI_ZSCORE'].notna().sum()
total_adolescent = mask_adolescent.sum()


print(f"   Adolescents (18-19)         : {total_adolescent:,}")
print(f"   BMI z-scores computed       : {computed:,}")
print(f"   Failed (missing BMI or age) : {total_adolescent - computed:,}")
print(f"\n   BMI z-score distribution (18-19 year-olds):")
print(f"   Mean   : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].mean():.3f}")
print(f"   Std    : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].std():.3f}")
print(f"   Min    : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].min():.3f}")
print(f"   Max    : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].max():.3f}")

Computing BMI Z-scores for 18-19 year-olds
All required cpolumns present
   Adolescents (18-19)         : 421
   BMI z-scores computed       : 0
   Failed (missing BMI or age) : 421

   BMI z-score distribution (18-19 year-olds):
   Mean   : nan
   Std    : nan
   Min    : nan
   Max    : nan


In [13]:
# Cell 11c FIXED: Compute BMI Z-score for adolescents
# Two issues found and fixed:
# 1. RIDAGEMN is NaN for all NHANES adults - we estimate age in months
# from RIDAGEYR instead (18 years ≈ 222.5 months, 19 years ≈ 234.5)
# 2. LMS Agemos column read as strings - we convert to float first

# Fix 1: Convert LMS table columns to correct types
lms['Agemos'] = pd.to_numeric(lms['Agemos'], errors='coerce')
lms['L']      = pd.to_numeric(lms['L'], errors='coerce')
lms['M']      = pd.to_numeric(lms['M'], errors='coerce')
lms['S']      = pd.to_numeric(lms['S'], errors='coerce')
lms['Sex']    = pd.to_numeric(lms['Sex'], errors='coerce')

# Drop any rows where Agemos became NaN (was a header artifact)
lms = lms.dropna(subset=['Agemos', 'L', 'M', 'S'])

print(f"LMS table cleaned:")
print(f"Agemos range: {lms['Agemos'].min():.1f} – {lms['Agemos'].max():.1f}")
print(f"Rows: {len(lms)}")

# Fix 2: Estimate age in months from RIDAGEYR
# NHANES suppresses RIDAGEMN for adults (privacy protection)
# We use the midpoint of each age year as the best available estimate
# 18 years = 216–227 months : midpoint 222.5
# 19 years = 228–239 months : midpoint 234.5
# LMS values change very slowly at this age so midpoint is sufficient

AGE_TO_MONTHS = {18: 222.5, 19: 234.5}

def compute_bmi_zscore_fixed(bmi, age_years, sex, lms_table):
    """
    Compute CDC BMI-for-age z-score using LMS method.
    Uses estimated age in months from whole-year RIDAGEYR.
    """
    if pd.isna(bmi) or pd.isna(age_years) or pd.isna(sex):
        return np.nan

    age_months = AGE_TO_MONTHS.get(int(age_years), None)
    if age_months is None:
        return np.nan

    # Find closest Agemos in LMS table for this sex
    sex_lms = lms_table[lms_table['Sex'] == int(sex)]
    closest_idx = (sex_lms['Agemos'] - age_months).abs().idxmin()
    row = sex_lms.loc[closest_idx]

    L, M, S = row['L'], row['M'], row['S']

    # Apply LMS formula (confirmed from CDC documentation)
    if L != 0:
        z = (((bmi / M) ** L) - 1) / (L * S)
    else:
        z = np.log(bmi / M) / S

    return z


# Apply to 18-19 year-olds
cohort['BMI_ZSCORE'] = np.nan
mask_adolescent = (cohort['RIDAGEYR'] >= 18) & (cohort['RIDAGEYR'] <= 19)

for idx in cohort[mask_adolescent].index:
    bmi      = cohort.loc[idx, 'BMXBMI']
    age_yrs  = cohort.loc[idx, 'RIDAGEYR']
    sex      = cohort.loc[idx, 'RIAGENDR']
    cohort.loc[idx, 'BMI_ZSCORE'] = compute_bmi_zscore_fixed(
        bmi, age_yrs, int(sex), lms
    )

# Report
computed = cohort.loc[mask_adolescent, 'BMI_ZSCORE'].notna().sum()
total    = mask_adolescent.sum()

print(f"\n BMI z-score results (18-19 year-olds):")
print(f"   Total adolescents           : {total:,}")
print(f"   Z-scores computed           : {computed:,}")
print(f"   Failed (missing BMI)        : {total - computed:,}")
print(f"\n   Distribution:")
print(f"   Mean : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].mean():.3f}")
print(f"   Std  : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].std():.3f}")
print(f"   Min  : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].min():.3f}")
print(f"   Max  : {cohort.loc[mask_adolescent, 'BMI_ZSCORE'].max():.3f}")

LMS table cleaned:
Agemos range: 24.0 – 240.5
Rows: 438

 BMI z-score results (18-19 year-olds):
   Total adolescents           : 421
   Z-scores computed           : 409
   Failed (missing BMI)        : 12

   Distribution:
   Mean : 0.606
   Std  : 1.299
   Min  : -3.766
   Max  : 3.426


In [16]:
# Cell 11d: Load MSSS calculator from scr
# Loads compute_msss() from the validated msss_calculator.py module.
# This keeps the formula logic separate from the notebook. Any future
# coefficient updates only require changing the .py file, not the notebook.

import importlib.util
# load the module from drive/src
src_dir = '/content/drive/MyDrive/mets-risk-score/src/'
module_path = os.path.join(src_dir, 'msss_calculator.py')

spec = importlib.util.spec_from_file_location('msss_calculator', module_path)
msss_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(msss_mod)

compute_msss = msss_mod.compute_msss
RACE_MAP = msss_mod.RACE_MAP

# Verify with a known complete adult row
# Selecting: Non-hispanic white female, age 20+, all 5 inpouts present
test_row = cohort[
    (cohort['RIDAGEYR']       >= 20)  &
    (cohort['RIDRETH3']       == 3.0) &   # Non-Hispanic White
    (cohort['RIAGENDR']       == 2.0) &   # Female
    (cohort['BMXWAIST'].notna())       &
    (cohort['LBXTR'].notna())          &
    (cohort['LBDHDD'].notna())         &
    (cohort['LBXGLU'].notna())         &
    (cohort['SYSTOLIC_AVG'].notna())
].iloc[0]

test_result = compute_msss(test_row)
is_valid    = not pd.isna(test_result)

print('MSSS calculator loaded')
print(f'Module: {module_path}')
print(f'Test: White Female age {test_row["RIDAGEYR"]:.0f} '
      f'MSSS = {test_result:.4f}')
print(f'Status: {"Valid result" if is_valid else "Returned NaN - check inputs"}')


MSSS calculator loaded
Module: /content/drive/MyDrive/mets-risk-score/src/msss_calculator.py
Test: White Female age 44 MSSS = -1.1381
Status: Valid result


In [21]:
# Cell 11e: Compute MSSS Target Variable

# Applies compute_msss() row-wise across the full 3750 person cohort
# Expected outcomes per participant:
# White / Black / Hispanic + complete inputs  -  real MSSS value
# Non-Hispanic Asian or Other/Multiracial     -  NaN (no equation)
# Any group with missing required inputs      -  NaN
#
# Required inputs per row:
# Adults (age >= 20) : BMXWAIST, LBDHDD, SYSTOLIC_AVG, LBXTR, LBXGLU
# Adolescents (<20)  : BMI_ZSCORE, LBDHDD, SYSTOLIC_AVG, LBXTR, LBXGLU


# Apply the formula row-wise
cohort['msss'] = cohort.apply(compute_msss, axis=1)

# Summary counts
n_total = len(cohort)
n_computed = cohort['msss'].notna().sum()
n_excluded = cohort[cohort['RIDRETH3'].isin([6.0, 7.0])].shape[0]
n_missing = n_total - n_computed - n_excluded

print(f'Total cohort: {n_total:,}')
print(f'MSSS computed: {n_computed:,}')
print(f'Excluded  (Asian/Other): {n_excluded:,}   no validated equation')
print(f'Missing   (incomplete labs): {n_missing:,}  fasting subsample gap')

# Distribution
msss_vals = cohort['msss'].dropna()

print(f'\n MSSS Distribution (n={n_computed:,}):')
print(f'Mean   : {msss_vals.mean():.3f}')
print(f'Std    : {msss_vals.std():.3f}')
print(f'Median : {msss_vals.median():.3f}')
print(f'Min    : {msss_vals.min():.3f}')
print(f'Max    : {msss_vals.max():.3f}')

# By sex
for code, label in [(1.0, 'Male'), (2.0, 'Female')]:
    s = cohort[cohort['RIAGENDR'] == code]['msss']
    print(f'   {label:<8} : n={s.notna().sum():,}  '
          f'mean={s.mean():.3f}  std={s.std():.3f}')

# By Race/Ethnicity
race_labels = {
    1.0: 'Mexican American',
    2.0: 'Other Hispanic',
    3.0: 'Non-Hispanic White',
    4.0: 'Non-Hispanic Black',
    6.0: 'Non-Hispanic Asian',
    7.0: 'Other/Multiracial',
}
for code, label in race_labels.items():
    s = cohort[cohort['RIDRETH3'] == code]['msss']
    if code in [6.0, 7.0]:
        print(f'   {label:<22} : excluded : no validated equation')
    else:
        print(f'   {label:<22} : '
              f'n={s.notna().sum():,}  mean={s.mean():.3f}')


Total cohort: 3,750
MSSS computed: 1,203
Excluded  (Asian/Other): 717   no validated equation
Missing   (incomplete labs): 1,830  fasting subsample gap

 MSSS Distribution (n=1,203):
Mean   : 0.010
Std    : 1.104
Median : -0.097
Min    : -2.070
Max    : 7.419
   Male     : n=594  mean=0.018  std=1.076
   Female   : n=609  mean=0.003  std=1.130
   Mexican American       : n=232  mean=0.118
   Other Hispanic         : n=138  mean=-0.121
   Non-Hispanic White     : n=462  mean=0.001
   Non-Hispanic Black     : n=371  mean=0.004
   Non-Hispanic Asian     : excluded : no validated equation
   Other/Multiracial      : excluded : no validated equation


In [29]:
# Cell 11f: Predictor/Target Split - Save to data/processed/

# produces the final model-ready dataset:

#   y (target)     →  continuous MSSS score
#   X (predictors) →  all remaining variables after leakage removal
#
# WHY WE REMOVE THE DIAGNOSTIC COMPONENTS:
#   The five MetS diagnostic components (waist, BP, triglycerides,
#   HDL, glucose) were used to COMPUTE msss. If left in X, the model
#   would learn to reverse-engineer the MSSS formula rather than
#   finding genuine signal in secondary biochemistry and lifestyle data.
#   Removing them is the core design decision that makes the prediction
#   task non-circular and clinically meaningful.
#
# COLUMNS EXCLUDED FROM X:
#   msss                        - target itself
#   SEQN                        - participant ID (not a predictor)
#   BMXWAIST                    - waist circumference (diagnostic)
#   BPXOSY1, BPXOSY2            - raw systolic BP readings (diagnostic)
#   BPXODI1, BPXODI2            - raw diastolic BP readings (diagnostic)
#   SYSTOLIC_AVG, DIASTOLIC_AVG - averaged BP (derived from diagnostic)
#   LBXTR, LBDTRSI              - triglycerides mg/dL + SI (diagnostic)
#   LBDHDD, LBDHDDSI            - HDL cholesterol mg/dL + SI (diagnostic)
#   LBXGLU, LBDGLUSI            - fasting glucose mg/dL + SI (diagnostic)
#   BMI_ZSCORE                  - used in adolescent MSSS formula

# Step 1: Filter to participants with valid MSSS
modeling_df = cohort[cohort['msss'].notna()].copy()
print(f"Participants with valid MSSS: {len(modeling_df):,}")

# Step 2: Define columns to remove
cols_to_remove = [
    'msss',
    'SEQN',
    'BMXWAIST',
    'BPXOSY1',      'BPXOSY2',
    'BPXODI1',      'BPXODI2',
    'SYSTOLIC_AVG', 'DIASTOLIC_AVG',
    'LBXTR',        'LBDTRSI',
    'LBDHDD',       'LBDHDDSI',
    'LBXGLU',       'LBDGLUSI',
    'BMI_ZSCORE',
]

# Only drop columns that actually exist in the dataframe
cols_to_remove = [c for c in cols_to_remove if c in modeling_df.columns]

# Step 3: Creat X and Y
y = modeling_df['msss'].copy()
x = modeling_df.drop(columns = cols_to_remove).copy()

print(f'\n Target (y:)')
print(f'Shape    : {y.shape}')
print(f'Mean     : {y.mean():.3f}')
print(f'Std      : {y.std():.3f}')
print(f'Min      : {y.min():.3f}')
print(f'Max      : {y.max():.3f}')

print(f'\n Predictors (x):')
print(f'Shape              : {x.shape}')
print(f'Columns removed    : {len(cols_to_remove)}')
print(f'Columns remaining  : {x.shape[1]}')

# Step 4: Missingness report
overall_missing = x.isnull().sum().sum()/(x.shape[0]*x.shape[1]) * 100
cols_with_missing = (x.isnull().sum() > 0).sum()

print(f'Overall          : {overall_missing:.1f}% of all values')
print(f'Affected columns : {cols_with_missing} of {x.shape[1]}')

# Step 5: Save all three files to data/processed/
x_path    = os.path.join(data_proc, 'x_modeling.csv')
y_path    = os.path.join(data_proc, 'y_msss.csv')
full_path = os.path.join(data_proc, 'modeling_cohort.csv')

x.to_csv(x_path,    index=False)
y.to_csv(y_path,    index=False)
modeling_df.to_csv(full_path, index=False)

print(f'x_modeling.csv       : {x.shape[0]:,} rows × {x.shape[1]:,} columns')
print(f'y_msss.csv           : {y.shape[0]:,} rows')
print(f'modeling_cohort.csv  : {modeling_df.shape[0]:,} rows × '
      f'{modeling_df.shape[1]:,} columns')

Participants with valid MSSS: 1,203

 Target (y:)
Shape    : (1203,)
Mean     : 0.010
Std      : 1.104
Min      : -2.070
Max      : 7.419

 Predictors (x):
Shape              : (1203, 435)
Columns removed    : 16
Columns remaining  : 435
Overall          : 25.4% of all values
Affected columns : 381 of 435
x_modeling.csv       : 1,203 rows × 435 columns
y_msss.csv           : 1,203 rows
modeling_cohort.csv  : 1,203 rows × 451 columns
